# Goal

Повторный прогон `18n_study_21.1`, в котором был ВЫКЛЮЧЕН prediction_loss. А нужно было с включенным.

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
    HP.encoder.is_trainable = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 1 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.0)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results

По сравнению с `18n_study_20.1` стат. значимых изменений не видно. 

> No, there is no statistically reliable difference between these two configurations. While experiment 18n_study_20.1 maintains its lead, the overlap between the two sets is highly pronounced, and the statistical tests firmly state that any differences could easily be down to random seed noise.


Получается, что экшн план роли особой не играет

| Metric | Experiment `18n_study_28.1` | Experiment `18n_study_20.1` | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- |
| **Sample Size (n)** | 24 | 24 | — |
| **Mean Score** | 0.5810 | 0.6185 | +0.0375 performance shift favoring `20.1` |
| **Standard Deviation** | 0.0986 | 0.0997 | Both groups exhibit nearly identical noise profiles |
| **Shapiro-Wilk (p-value)** | 0.1425 (Normal) | 0.0594 (Normal) | Both datasets follow a standard bell curve distribution |
| **Welch's t-test (p-value)** | — | — | **0.1966** (Not Significant; well above α = 0.05) |
| **Mann-Whitney U (p-value)** | — | — | **0.2432** (Not Significant; rank-order matches the t-test verdict) |
| **Cohen's d (Effect Size)** | — | — | **-0.3783** (Small-to-medium magnitude favoring `20.1`) |

**Выводы**
1) сделать суперисследование на 2-ой и 3-ий шаг, чтобы окончательно укрепить или опровергнуть заключение о ненужности экшн плана

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_28.1/18n_study_28.1.ipynb',
 'optuna_study_name': '18n_study_28.1',
 'optuna_study_serial': '28.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_28.1/18n_study_28.1.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [10]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 24
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-09-29 10:40:16,000] A new study created in Journal with name: 18n_study_28.1


2026.09.29-10:40:16.293753     0.262 >> Model instance registered, version=352
2026.09.29-10:40:16.308983     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch352.ipynb"
2026.09.29-10:40:16.309278     0.006 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:352"; running launches=1
2026.09.29-10:40:47.494972     0.255 >> Model instance registered, version=353
2026.09.29-10:40:47.503981     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch353.ipynb"
2026.09.29-10:40:47.504233     0.004 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:353"; running launches=2
2026.09.29-10:41:18.433610     0.238 >> Model instance registered, version=354
2026.09.29-10:41:18.449515     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch354.ipynb"
2026.09.29-10:41:18.449832     0.006 >> 4.2 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-09-29 14:06:45,651] Using an existing study with name '18n_study_28.1' instead of creating a new one.


2026.09.29-14:06:45.652904     5.034 >> Study statistics: 
2026.09.29-14:06:45.656120     0.003 >> 	Number of finished trials: 24
2026.09.29-14:06:45.657224     0.001 >> 	Number of pruned trials: 0
2026.09.29-14:06:45.658158     0.001 >> 	Number of complete trials: 24
2026.09.29-14:06:45.658897     0.001 >> Best trial:
2026.09.29-14:06:45.660585     0.002 >> 	Value: 0.778125
2026.09.29-14:06:45.661057     0.000 >> 	Model version: 369
2026.09.29-14:06:45.661384     0.000 >> 	Params: 


In [23]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.09.29-16:56:11.625642 10165.964 >> Top trials:
2026.09.29-16:56:11.626383     0.001 >> 1) 18n_ppo_tr_frostbite_07:369, value=0.778125
2026.09.29-16:56:11.626765     0.000 >> 2) 18n_ppo_tr_frostbite_07:367, value=0.75
2026.09.29-16:56:11.627095     0.000 >> 3) 18n_ppo_tr_frostbite_07:354, value=0.7375


In [12]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(0.5809895833333333), np.float64(0.0965001006359219))

In [21]:
','.join(map(str, values.tolist()))

'0.596875,0.571875,0.7375,0.453125,0.446875,0.471875,0.484375,0.596875,0.5125,0.7,0.45,0.60625,0.5375,0.571875,0.596875,0.75,0.5,0.778125,0.7,0.659375,0.496875,0.59375,0.534375,0.596875'

In [18]:
study_20_1 = optuna.create_study(
    study_name='18n_study_20.1',
    storage=JournalStorage(JournalFileBackend(file_path=os.path.join(os.path.abspath('..'), '18n_study_20', 'studies', '18n_study_20.1.optuna'))),
    load_if_exists=True, 
)

values_20_1 = []

for trial in study_20_1.trials:
    if trial.values is not None:
        values_20_1.append(trial.values[0])

values_20_1 = np.array(values_20_1)

[I 2026-09-29 16:41:10,823] Using an existing study with name '18n_study_20.1' instead of creating a new one.


In [22]:
','.join(map(str, values_20_1.tolist()))

'0.696875,0.534375,0.5875,0.790625,0.759375,0.546875,0.796875,0.546875,0.503125,0.575,0.578125,0.596875,0.49375,0.4875,0.671875,0.596875,0.68125,0.596875,0.5875,0.8375,0.6375,0.5,0.590625,0.65'

In [20]:
import scipy.stats as stats
import numpy as np
import pandas as pd
from collections import defaultdict

columns = defaultdict(list)
columns['name'].extend(['28.1 (current)', '20.1 (prev)', '-'])

# Basic metrics
values_28_1 = values
mean_28_1 = np.mean(values_28_1)
mean_20_1 = np.mean(values_20_1)
columns['mean'].extend([mean_28_1, mean_20_1, None])

std_28_1 = np.std(values_28_1, ddof=1)
std_20_1 = np.std(values_20_1, ddof=1)
columns['std'].extend([std_28_1, std_20_1, None])

# Normality Check (Shapiro-Wilk)
shapiro_28_1 = stats.shapiro(values_28_1)
shapiro_20_1 = stats.shapiro(values_20_1)
columns['shapiro'].extend([shapiro_28_1.pvalue, shapiro_20_1.pvalue, None])

# Welch's t-test
t_stat, p_welch = stats.ttest_ind(values_28_1, values_20_1, equal_var=False)
columns['welch'].extend([None, None, p_welch])

# Mann-Whitney U test
u_stat, p_mw = stats.mannwhitneyu(values_28_1, values_20_1, alternative='two-sided')
columns['mann-whitney'].extend([None, None, p_mw])

# Cohen's d
pooled_std = np.sqrt(((len(values_28_1)-1)*std_28_1**2 + (len(values_20_1)-1)*std_20_1**2) / (len(values_28_1) + len(values_20_1) - 2))
cohens_d = (mean_28_1 - mean_20_1) / pooled_std
columns['cohen d'].extend([None, None, cohens_d])

pd.DataFrame(columns)

,name,mean,std,shapiro,welch,mann-whitney,cohen d
0,28.1 (current),0.58099,0.098576,0.142544,NaN,NaN,NaN
1,20.1 (prev),0.61849,0.099689,0.059375,NaN,NaN,NaN
2,-,NaN,NaN,NaN,0.196571,0.24322,-0.378276
